# 01 — Dataset audit and acquisition plan
The default run audits the original 10,000-row SYNTHETIC dataset. Its plots are data diagnostics, not model results or real-5G evidence. No row-order sequences are created.

For real data we first inspect provider metadata and licenses. CESNET DataZoo S contains 25 million samples: do not treat it as a small Colab download. Candidate sources and their limitations are in research/README.md.


In [ ]:
from pathlib import Path
import subprocess, sys, json, datetime, platform, importlib.metadata
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/5G_QoS_Research')
for name in ['environment', 'audits', 'data/manifests', 'configs', 'results', 'models', 'figures']:
    (PROJECT / name).mkdir(parents=True, exist_ok=True)
print('Persistent project:', PROJECT)


In [ ]:
REPO = Path('/content/ai-qos-research')
URL = 'https://github.com/Benzsoft/ai-qos-classification-5g.git'
REF = 'research/colab-foundation'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', REF, '--single-branch', URL, str(REPO)], check=True)
else:
    origin = subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip()
    assert origin == URL, f'Unexpected repository: {origin}'
    print('Reusing checkout without overwriting local work.')
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Exact research commit:', COMMIT)


In [ ]:
import sys
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from research.audit import audit_csv
# Replace only with a local CSV path and an explicitly identified label column.
CSV_PATH = REPO / 'src/synthetic_5g_qos_dataset_10000.csv'
LABEL = 'qos_class'
PROVENANCE = 'repository_synthetic'
MAX_ROWS = 100_000  # Bounded prefix audit, not a representative sample or training split.
report, output = audit_csv(CSV_PATH, PROJECT / 'audits', label=LABEL,
                          provenance=PROVENANCE, max_rows=MAX_ROWS)
print(json.dumps(report, indent=2))
print('Saved diagnostics:', output)


In [ ]:
from IPython.display import display, Image
import pandas as pd
display(pd.read_csv(output / 'class_counts.csv'))
display(pd.read_csv(output / 'schema.csv'))
display(Image(filename=str(output / 'class_counts.png')))


## Next data gate
A real-data adapter requires: provider/version/license; actual file listing and sizes; label provenance; timestamp units; flow/session boundaries; packet direction; identifier availability; and an acquisition checksum. Never infer URLLC from voice or mMTC from small packet sizes.

Application labels and simulated service profiles are separate targets. Fit preprocessing only on training partitions. Split sessions/time/scenarios before constructing windows. IP embeddings are evaluated only when legitimate identifiers are available.


In [ ]:
manifest = {
    'status': 'candidate_not_downloaded_or_approved_for_training',
    'candidates': [
        {'name':'Korean 5G Traffic Datasets', 'url':'https://www.kaggle.com/datasets/kimdaegyeom/5g-traffic-datasets', 'purpose':'real 5G application traffic', 'pending':['file inventory','license','session and label audit']},
        {'name':'CESNET-TLS-Year22', 'url':'https://cesnet.github.io/cesnet-datazoo/', 'purpose':'temporal shift in backbone service traffic; not inherently 5G', 'pending':['version and license','bounded acquisition strategy','identifier availability']},
        {'name':'ISCXVPN2016', 'url':'https://www.unb.ca/cic/datasets/vpn.html', 'purpose':'older application-traffic pilot', 'pending':['download access','license','PCAP/session labels']}
    ]
}
manifest_path = PROJECT / 'data/manifests' / 'candidate_datasets.json'
if not manifest_path.exists():
    manifest_path.write_text(json.dumps(manifest, indent=2))
else:
    print('Preserving existing candidate manifest.')
print('Audit finished. Share the printed JSON report, not raw traffic or credentials.')
